# toro-CM1 no vlab — Lhende Khola, 25–26/08/2026

Roda as 13 rodadas do CM1 (`TORO_CM1_SONDAGEM_REAL_ERA5.md`) pelo Slurm, a partir do Jupyter (`jupyter.vlab.ufsc.br`), **sem scp/rsync**.

**Fluxo com o seu computador: cache no Google Drive.** No Windows, a pasta `G:\Meu Drive\vlab_cache\` (Google Drive para desktop): `entrada\` vai para o vlab, `saida\` volta dele. No vlab, o `rclone` acessa o mesmo Drive.

Na primeira vez o pacote vem do GitHub, que é público; depois, tudo passa pelo Drive. Rode as células em ordem.

## 1. Baixar e conferir o pacote
**1a — primeira vez (GitHub).** Depois de configurar o Drive (célula 1b), use `bash gdrive.sh baixar` para pegar versões novas que você puser em `G:\Meu Drive\vlab_cache\entrada\`.

In [ ]:
%%bash -l
mkdir -p $HOME/mesoescala-montanhas && cd $HOME/mesoescala-montanhas
URL=https://raw.githubusercontent.com/reinaldohaas/Himalaia_2026/refs/heads/claude/himalaia-enchentes-auditoria-fcfdb5/cluster/toro_cm1_lhende_vlab.tar.gz
wget -q -O toro_cm1_lhende_vlab.tar.gz "$URL" && wget -q -O toro_cm1_lhende_vlab.tar.gz.sha256 "$URL.sha256"
sha256sum -c toro_cm1_lhende_vlab.tar.gz.sha256
[ -d toro_cm1_lhende ] && echo "toro_cm1_lhende ja existe (nao sobrescrito)" || tar xzf toro_cm1_lhende_vlab.tar.gz
cd toro_cm1_lhende && sha256sum -c --quiet MANIFESTO.sha256 && echo "manifesto OK"


**1b — configurar o Google Drive (uma vez).** A célula instala o rclone e mostra o passo a passo. A configuração em si é interativa: faça-a num **Terminal** do Jupyter (*File → New → Terminal*) com `rclone config`. O login do Google é seu, pelo `rclone authorize` no seu computador; o token fica só no `rclone.conf` do vlab.

In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
bash gdrive.sh instalar
bash gdrive.sh configurar


In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
bash gdrive.sh estado      # depois de configurado


## 2. Ver o cluster (partições, compilador, módulos)
Anote o nome da partição de CPU para a célula 4.

In [ ]:
%%bash -l
sinfo -s
command -v gfortran && gfortran --version | head -1 || echo "gfortran nao esta no PATH"
module avail 2>&1 | grep -iE 'gcc|gnu|openmpi' | head || true


## 3. Compilar (uma vez, ~2–5 min)
Usa o gfortran do sistema, o módulo `gcc` ou cria o ambiente conda `cm1` na `$HOME`.

In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
NPROC=8 bash compilar.sh


## 4. Teste curto (120 s de modelo) e submissão
Troque `PART` pela partição vista na célula 2. Primeiro o teste; se o log terminar com `Program terminated normally`, submeta as rodadas.

In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
PART=cpu          # <<< ajuste
bash submeter.sh -t -p $PART rad_12utc


In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
tail -n 5 logs/cm1_rad_12utc_teste-*.out


In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
PART=cpu          # <<< ajuste
bash submeter.sh -p $PART rad        # as duas com radiacao (rad_06utc e a principal)
# bash submeter.sh -p $PART todos    # ou as 13


## 5. Acompanhar

In [ ]:
%%bash -l
squeue -u $USER
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
for f in logs/cm1_*.out; do echo "== $f"; tail -n 2 "$f"; done 2>/dev/null
for d in execucoes/*/; do printf '%-28s ' $(basename $d); tail -n 1 $d/execucao.info 2>/dev/null || echo '(rodando)'; done 2>/dev/null


## 6. Análise e coleta leve
Gera figuras e JSON em `pos/` e um pacote de poucos MB (`resultados_cm1_leve_*.tar.gz`). As saídas 3D (~11 GB) ficam no cluster.

In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
bash pos/analisar.sh
bash coletar.sh --leve


## 7. Devolver ao seu computador pelo Google Drive
Sobe o pacote leve e as figuras para `vlab_cache/saida/toro_cm1_lhende/<data>/`; aparecem em `G:\Meu Drive\vlab_cache\saida\...` no Windows.

Com `--completo`, sobe também o pacote 3D completo (~11 GB, antes gere-o com `bash coletar.sh`). Isso consome cota do seu Drive.

Alternativa sem Drive: `bash devolver_github.sh` (GitHub Release, público).

In [ ]:
%%bash -l
cd $HOME/mesoescala-montanhas/toro_cm1_lhende
bash gdrive.sh enviar
# bash coletar.sh && bash gdrive.sh enviar --completo
